In [1]:
%pip install plotly

   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   --------------------------------- ------ 12.3/14.8 MB 64.1 MB/s eta 0:00:01
   ---------------------------------------- 14.8/14.8 MB 58.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install tqdm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonschema-4.23.0-py3-none-any.whl.metadata (7.9 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.1-py3-none-any.whl (23 kB)
Using cached jsonschema-4.23.0-py3-none-any.whl (88 kB)
Note: you may need to restart the kernel to use updated packages.


In [120]:
from collections import Counter

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm

In [2]:
agenda_items = pd.read_csv('../data/currentTermAgendaItems.csv')

## Exploring all fields first...

In [3]:
agenda_items.head(5)

,id,termId,agendaItemId,councilAgendaItemId,decisionBodyId,meetingId,itemProcessId,decisionBodyName,meetingDate,reference,...,decisionAdvice,subjectTerms,wardId,backgroundAttachmentId,agendaItemAddress,address,geoLocation,planningApplicationNumber,neighbourhoodId,textSearchVector
0,b1b96e68-de7f-487a-b849-a844e49316d8,8,137552,137552,2643,24801,6,Sign Variance Committee,1731646800000,2024.SB13.2,...,NaN,;,NaN,NaN,[],NaN,NaN,NaN,NaN,'committe':3A 'sign':1A 'varianc':2A
1,72dcc5e7-a826-4c23-8b50-d477740701a1,8,127615,127615,2542,23202,6,General Government Committee,1677819600000,2023.GG2.28,...,NaN,by-laws; bylaws,"[45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,...",NaN,[],NaN,NaN,NaN,NaN,'committe':3A 'general':1A 'govern':2A
2,ee552860-e13e-4c85-857b-e37c05860914,8,137462,137462,2489,24917,6,Toronto Zoo,1731906000000,2024.ZB13.1,...,"<p>Dr. Gabriela Mastromonaco, Senior Director,...","conservation, zoo animals, zoos; environmental...","[45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,...",[250740],[],NaN,NaN,NaN,NaN,'toronto':1A 'zoo':2A
3,8ffcecb7-22dc-4294-a33b-c765781592e0,8,129735,129735,2587,23426,6,Toronto Atmospheric Fund,1689307200000,2023.TA3.7,...,NaN,minutes;,NaN,[238037],[],NaN,NaN,NaN,NaN,'atmospher':2A 'fund':3A 'toronto':1A
4,9d000738-fd59-4635-a786-7ab1976f0d29,8,137528,137528,2542,24418,6,General Government Committee,1732078800000,2024.GG18.15,...,NaN,"purchasing, service delivery, water; acquisiti...","[45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,...",[250086],[],NaN,NaN,NaN,NaN,'committe':3A 'general':1A 'govern':2A


In [4]:
agenda_items.columns

Index(['id', 'termId', 'agendaItemId', 'councilAgendaItemId', 'decisionBodyId',
       'meetingId', 'itemProcessId', 'decisionBodyName', 'meetingDate',
       'reference', 'termYear', 'agendaCd', 'meetingNumber', 'itemStatus',
       'agendaItemTitle', 'agendaItemSummary', 'agendaItemRecommendation',
       'decisionRecommendations', 'decisionAdvice', 'subjectTerms', 'wardId',
       'backgroundAttachmentId', 'agendaItemAddress', 'address', 'geoLocation',
       'planningApplicationNumber', 'neighbourhoodId', 'textSearchVector'],
      dtype='object')

In [5]:
np.sum(agenda_items['termId'].apply(lambda x: int(pd.isna(x))))

np.int64(0)

In [ ]:
# Count the number of nulls / NaNs
counts = {}
for col in agenda_items.columns:
    counts[col] = np.sum(agenda_items[col].apply(lambda x: int(pd.isna(x)))).item()
    # next time: could just do agenda_items.shape[0] - agenda_items[col].count()
counts

{'id': 0,
 'termId': 0,
 'agendaItemId': 0,
 'councilAgendaItemId': 0,
 'decisionBodyId': 0,
 'meetingId': 0,
 'itemProcessId': 0,
 'decisionBodyName': 0,
 'meetingDate': 0,
 'reference': 0,
 'termYear': 0,
 'agendaCd': 10,
 'meetingNumber': 0,
 'itemStatus': 0,
 'agendaItemTitle': 0,
 'agendaItemSummary': 0,
 'agendaItemRecommendation': 1342,
 'decisionRecommendations': 205,
 'decisionAdvice': 8714,
 'subjectTerms': 0,
 'wardId': 382,
 'backgroundAttachmentId': 444,
 'agendaItemAddress': 0,
 'address': 6114,
 'geoLocation': 6114,
 'planningApplicationNumber': 9429,
 'neighbourhoodId': 9362,
 'textSearchVector': 0}

In [7]:
agenda_items.shape

(10412, 28)

In [71]:
normalized_counts = {field: counts[field] / agenda_items.shape[0] * 100 for field in counts}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_counts],
        y=[normalized_counts[field] for field in normalized_counts],
        # textposition='inside',
        text=[str(round(normalized_counts[field], 1)) for field in normalized_counts],
        # text_auto="True"
        # title='Missingness of Agenda Item Fields',
    )],
    layout=go.Layout(
        title={'text': 'Missingness of Agenda Item Fields', 'subtitle': {'text': 'Or, the amount of null values.'}},
        # subtitle_text='Or, the amount of null values',
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}, 'range': [0, 105]}
    )
    # layout_title_text='Missingness of Agenda Item Fields',
    # x_axis={'title': {'text': 'Field'}}
    
)

In [72]:
fig

In [39]:
unique_counts = {}
for field in agenda_items.columns:
    unique_counts[field] = len(set(agenda_items[field]))
unique_counts


{'id': 10412,
 'termId': 1,
 'agendaItemId': 8494,
 'councilAgendaItemId': 10412,
 'decisionBodyId': 69,
 'meetingId': 832,
 'itemProcessId': 6,
 'decisionBodyName': 69,
 'meetingDate': 427,
 'reference': 8494,
 'termYear': 3,
 'agendaCd': 77,
 'meetingNumber': 120,
 'itemStatus': 15,
 'agendaItemTitle': 7176,
 'agendaItemSummary': 7606,
 'agendaItemRecommendation': 6638,
 'decisionRecommendations': 9667,
 'decisionAdvice': 1351,
 'subjectTerms': 4169,
 'wardId': 229,
 'backgroundAttachmentId': 8053,
 'agendaItemAddress': 3304,
 'address': 2711,
 'geoLocation': 2679,
 'planningApplicationNumber': 483,
 'neighbourhoodId': 237,
 'textSearchVector': 69}

In [ ]:
np.sum(agenda_items['agendaCd'].apply(lambda x: int(not pd.isna(x))))

np.int64(10402)

In [49]:
normalized_uniqueness = {field: unique_counts[field] / agenda_items.shape[0] * 100 for field in counts}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_uniqueness],
        y=[normalized_uniqueness[field] for field in normalized_uniqueness],
        text=[str(round(normalized_uniqueness[field], 1)) for field in normalized_uniqueness],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Agenda Item Fields'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}}
    )
)

In [50]:
fig

In [62]:
unique_counts_excl_na = {}
for field in agenda_items.columns:
    # Gather unique values
    unique_values = set(agenda_items[field])
    unique_values.discard(np.nan)
    
    # Field: (# unique non-null values, total # non-null values)
    unique_counts_excl_na[field] = len(unique_values), agenda_items.shape[0] - counts[field]
unique_counts_excl_na


{'id': (10412, 10412),
 'termId': (1, 10412),
 'agendaItemId': (8494, 10412),
 'councilAgendaItemId': (10412, 10412),
 'decisionBodyId': (69, 10412),
 'meetingId': (832, 10412),
 'itemProcessId': (6, 10412),
 'decisionBodyName': (69, 10412),
 'meetingDate': (427, 10412),
 'reference': (8494, 10412),
 'termYear': (3, 10412),
 'agendaCd': (76, 10402),
 'meetingNumber': (120, 10412),
 'itemStatus': (15, 10412),
 'agendaItemTitle': (7176, 10412),
 'agendaItemSummary': (7606, 10412),
 'agendaItemRecommendation': (6637, 9070),
 'decisionRecommendations': (9666, 10207),
 'decisionAdvice': (1350, 1698),
 'subjectTerms': (4169, 10412),
 'wardId': (228, 10030),
 'backgroundAttachmentId': (8052, 9968),
 'agendaItemAddress': (3304, 10412),
 'address': (2710, 4298),
 'geoLocation': (2678, 4298),
 'planningApplicationNumber': (482, 983),
 'neighbourhoodId': (236, 1050),
 'textSearchVector': (69, 10412)}

In [64]:
normalized_uniqueness_excl_na = {field: unique_counts_excl_na[field][0] / unique_counts_excl_na[field][1] * 100 for field in unique_counts_excl_na}
fig = go.Figure(
    data=[go.Bar(
        x=[field for field in normalized_uniqueness_excl_na],
        y=[normalized_uniqueness_excl_na[field] for field in normalized_uniqueness_excl_na],
        text=[str(round(normalized_uniqueness_excl_na[field], 1)) for field in normalized_uniqueness_excl_na],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Agenda Item Fields, excluding null values'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}}
    )
)

In [65]:
fig

## Moving onto subject terms themselves...

In [84]:
agenda_items['subjectTerms'].count()

np.int64(10412)

In [95]:
subject_terms_semicolon_sep = []
subject_terms_comma_sep = []
for i in range(agenda_items.shape[0]):
    subject_terms_semicolon_sep += [value.strip() for value in agenda_items['subjectTerms'].iloc[i].split(';') if value.strip()]
for term in subject_terms_semicolon_sep:
    subject_terms_comma_sep += [value.strip() for value in term.split(',') if value.strip()]
print("len(semicolon-separated):", len(subject_terms_semicolon_sep))
print("len(semicolon-separated, unique):", len(set(subject_terms_semicolon_sep)))
print("len(comma-separated):", len(subject_terms_comma_sep))
print("len(comma-separated, unique):", len(set(subject_terms_comma_sep)))

len(semicolon-separated): 45573
len(semicolon-separated, unique): 8080
len(comma-separated): 65196
len(comma-separated, unique): 3230


In [81]:
subject_terms_semicolon_sep[1000:1010]

['laneways, transportation',
 'back alleys',
 'mews, transportation services',
 'rezoning, zoning bylaws',
 're-zoning',
 'zoning by-law amendment applications',
 'zoning bylaw amendments, zoning by-laws',
 'zoning bylaw project',
 'zoning project',
 'official plan amendments, rezoning, zoning bylaws']

In [94]:
with open("semicolon_separated_subject_terms.txt", 'w', encoding='utf-8') as file:
    for term in subject_terms_semicolon_sep:
        file.write(term)
        file.write("\n")
with open("comma_separated_subject_terms.txt", 'w', encoding='utf-8') as file:
    for term in subject_terms_comma_sep:
        file.write(term)
        file.write("\n")
with open("semicolon_separated_subject_terms_unique.txt", 'w', encoding='utf-8') as file:
    for term in sorted(set(subject_terms_semicolon_sep)):
        file.write(term)
        file.write("\n")
with open("comma_separated_subject_terms_unique.txt", 'w', encoding='utf-8') as file:
    for term in sorted(set(subject_terms_comma_sep)):
        file.write(term)
        file.write("\n")

In [77]:
agenda_items.iloc[10408]

id                                        8b667377-aef2-4dc5-9021-28cc7629b71b
termId                                                                       8
agendaItemId                                                            131768
councilAgendaItemId                                                     131768
decisionBodyId                                                            2466
meetingId                                                                23289
itemProcessId                                                                6
decisionBodyName                       Toronto and East York Community Council
meetingDate                                                      1700024400000
reference                                                          2023.TE9.84
termYear                                                                  2023
agendaCd                                                                    TE
meetingNumber                                       

In [ ]:
term_uniqueness = [
    len(set(subject_terms_semicolon_sep)) / len(subject_terms_semicolon_sep),
    len(set(subject_terms_comma_sep)) / len(subject_terms_comma_sep)
]
term_uniqueness = [val * 100 for val in term_uniqueness]

# Graph uniqueness
fig = go.Figure(
    data=[go.Bar(
        x=['Semicolon-separated terms', 'Comma-separated terms'],
        y=term_uniqueness,
        text=[str(round(val, 1)) for val in term_uniqueness],
    )],
    layout=go.Layout(
        title={'text': 'Uniqueness of Subject Terms'},
        xaxis={'title': {'text': 'Field'}},
        yaxis={'title': {'text': 'Percent'}, 'range': [0, 105]}
    )
)
fig

In [136]:
# Graph uniqueness
plot_df = pd.DataFrame(
    [
        ['Semicolon-separated', len(set(subject_terms_semicolon_sep)), len(subject_terms_semicolon_sep) - len(set(subject_terms_semicolon_sep))],
        ['Comma-separated', len(set(subject_terms_comma_sep)), len(subject_terms_comma_sep) - len(set(subject_terms_comma_sep))]
    ],
    columns=['Subject Terms', 'Unique', 'Duplicate'])
fig = px.bar(
    plot_df,
    x='Subject Terms',
    y=['Unique', 'Duplicate'],
    title='Uniqueness of Subject Terms',
    text_auto=True,
    # text=[{'Unique': 1, 'Duplicate': 2}, {'Unique': 3, 'Duplicate': 4}],
    height=600,
    width=550,
)
fig.show()

### Calculate overlap of subject terms with City Subject Thesaurus

In [96]:
city_subject_thesaurus = pd.read_csv('../data/City Subject Thesaurus (xls).csv')
city_subject_thesaurus

,Name,URI,Identifier,Definition(s),Source,Synonym(s),Status,Broader concept,Narrower concept,Related concept,Legacy ID
0,[actions of government],http://vocab.toronto.ca/id/100783,100783,placeholder,developed by editor,NaN,approved,[activities in government],"business travel (municipal), city initiatives,...",NaN,3926
1,[activities in business and industry],http://vocab.toronto.ca/id/100192,100192,placeholder,"Developed by editors. (Jan 21, 2008)",NaN,approved,business & industry (sc),"business registration, business start-up, cons...",NaN,3196
2,[activities in community and life],http://vocab.toronto.ca/id/101082,101082,placeholder,"Developed by editors. (Mar 12, 2008)",NaN,approved,community & life (sc),"adoption, child care, child custody, child sup...",NaN,3590
3,[activities in culture],http://vocab.toronto.ca/id/100188,100188,placeholder,"Developed by editors. (Mar 12, 2008)",NaN,approved,culture (sc),arts and culture,NaN,3581
4,[activities in education],http://vocab.toronto.ca/id/100933,100933,placeholder,"Developed by editors. (Mar 5, 2008)",NaN,approved,education (sc),"adult and community education, early childhood...","[activities in education], public education",3490
...,...,...,...,...,...,...,...,...,...,...,...
2507,zoning,http://vocab.toronto.ca/id/100603,100603,the government regulation of land and building...,From Access Toronto Kb document (Building  Zo...,NaN,approved,[activities in planning and development],"interim control, part lot control","rezoning, zoning, zoning bylaws, zoning design...",3338
2508,zoning bylaws,http://vocab.toronto.ca/id/100601,100601,"Municipal laws that regulate the use, size, he...",From Access Toronto Kb document (Building  Zo...,"zoning by-laws, zoning bylaw project, zoning p...",approved,[rules in planning and development],NaN,"interim control, land use, minor variances, of...",680
2509,zoning designations,http://vocab.toronto.ca/id/100607,100607,"the controls outlined in the zoning bylaw, for...","City web page, Toronto Building, Customer Serv...","permitted use requests, permitted uses",approved,[attributes in property],NaN,"land use, rezoning, zoning, zoning bylaws, zon...",684
2510,zoo animals,http://vocab.toronto.ca/id/101369,101369,placeholder,from Access Toronto Kb document (Parks - High ...,NaN,approved,[objects in recreation and tourism],NaN,"zoo animals, zoos",1123


In [97]:
len(set(city_subject_thesaurus['Name']))

2512

In [106]:
cst = set(city_subject_thesaurus['Name'])
comma_st = set(subject_terms_comma_sep)
semicolon_st = set(subject_terms_semicolon_sep)
print('cst comma semicolon')
print(len(cst), len(comma_st), len(semicolon_st))

cst comma semicolon
2512 3230 8080


In [101]:
len(comma_st & cst)

1198

In [102]:
len(comma_st & semicolon_st)

2083

In [103]:
len(semicolon_st & cst)

316

In [104]:
len(cst & semicolon_st & comma_st)

314

In [105]:
len(cst - semicolon_st - comma_st)

1312

In [108]:
1312+316+1198-314


2512

In [109]:
1198-314

884

In [110]:
2+314+884+1312

2512

In [111]:
2083-314

1769

In [112]:
len(semicolon_st & comma_st - (semicolon_st & comma_st & cst))

1769

In [114]:
len(comma_st - cst - semicolon_st)

263

In [115]:
len(semicolon_st - cst - comma_st)

5995

In [117]:
len(comma_st - cst)

2032

In [119]:
len(semicolon_st - comma_st)

5997